In [1]:
import numpy as np
import pandas as pd
import os, time, re, urllib.parse
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
from openlocationcode import openlocationcode as olc
global locations_data, referenceLatitude, referenceLongitude
locations_data = "csv-locations_12.9514242_77.6590212.csv"
referenceLatitude = float(locations_data.strip(".csv").split("_")[1])
referenceLongitude = float(locations_data.strip(".csv").split("_")[2])
locations_df = pd.read_csv(locations_data)
routes_df = pd.read_csv("csv-routes.csv")
out_file = "csv-bangalore_traffic"

In [3]:
# Selenium options required to create a 'headless' browser
options = Options()
options.add_argument("--blink-settings=imagesEnabled=false")
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1280,800")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.5481.77 Safari/537.37")
driver = webdriver.Chrome(options=options)

In [4]:
def is_plus_code(code):
    # Regex pattern for Plus Codes like "7FG9V3F5+X2" or "2HVW+G8"
    pattern = r'^[2-9A-HJ-NP-Z]{4,7}\+[2-9A-HJ-NP-Z]{2,}$'
    return re.match(pattern, code) is not None

def get_maps_url(origin, destination):
    origin = urllib.parse.quote(origin)
    destination = urllib.parse.quote(destination)
    url = f"https://www.google.com/maps/dir/?api=1&origin={origin}&destination={destination}&travelmode=driving"
    return url

def get_route_points(route_code, format="short"):
    origin, destination = route_code.split("|")

    if format in ["long", "latlong"]:
        origin = olc.recoverNearest(origin, referenceLatitude, referenceLongitude)
        destination = olc.recoverNearest(destination, referenceLatitude, referenceLongitude)

    if format == "latlong":
        origin = olc.decode(origin).latitudeCenter
        destination = olc.decode(destination).longitudeCenter

    return origin, destination

def get_traffic_report(origin, destination, mode='car', max_retries=3, retry_delay=60):
    modes = {'bike': "\ue9f9", 'car': "\ue531", 'transit': "\ue535"}
    if mode in modes.keys():
        mode = modes[mode]
    else:
        mode = modes['car']

    attempts = 0
    while True:
        try:
            maps_url = get_maps_url(origin, destination)
            print(f"From {origin} to {destination}\n{maps_url}")
            driver.get(maps_url)

            routes = driver.find_elements(By.CSS_SELECTOR, "div[data-trip-index]")

            # Initialize defaults from the first route; set None on IndexError
            try:
                parts0 = routes[0].text.split("\n")
                time_taken = parts0[1]
                distance = parts0[2]
            except IndexError:
                time_taken = None
                distance = None

            # Try to refine using the mode-specific route; set None on IndexError
            for route in routes:
                if mode in route.text:
                    parts = route.text.split("\n")
                    try:
                        time_taken = parts[1]
                    except IndexError:
                        time_taken = None
                    try:
                        distance = parts[2]
                    except IndexError:
                        distance = None
                    break

            return time_taken, distance

        except ValueError as e:
            attempts += 1
            if attempts >= max_retries:
                raise
            time.sleep(retry_delay)

        except Exception as e:
            attempts += 1
            if attempts >= max_retries:
                raise
            time.sleep(retry_delay)

def transformed_data(df_in):
    df_traffic = df_in.copy()
    df_traffic['year'] = pd.to_datetime(df_traffic['date']).dt.year
    df_traffic['month'] = pd.to_datetime(df_traffic['date']).dt.month
    df_traffic['date'] = pd.to_datetime(df_traffic['date']).dt.day
    df_traffic['hour'] = pd.to_datetime(df_traffic['time'], format='%H:%M', errors='coerce').dt.hour
    df_traffic['day_of_week'] = pd.to_datetime(df_traffic['date']).dt.day_name()
    df_traffic['avg_speed'] = round(df_traffic['distance'] / (df_traffic['duration'] / 60), 2)
    df_traffic['origin'] = df_traffic['route_code'].str.split('|').str[0]
    df_traffic['destination'] = df_traffic['route_code'].str.split('|').str[1]
    df_traffic['origin'] = df_traffic['origin'].map(locations_df.set_index('plus_code')['location'])
    df_traffic['destination'] = df_traffic['destination'].map(locations_df.set_index('plus_code')['location'])
    df_traffic = df_traffic[['year', 'month', 'date', 'hour', 'day_of_week', 'origin', 'destination', 'duration', 'distance', 'avg_speed']]
    df_traffic = df_traffic.sort_values('avg_speed', ascending=True).reset_index(drop=True)
    return df_traffic


In [5]:
df = pd.DataFrame()
date_now = datetime.now().date()
time_now = datetime.now().time().strftime("%H:%M")

for index, route in routes_df.iterrows():
    origin, destination = get_route_points(route["route_code"])
    origin = locations_df[locations_df["plus_code"] == origin]["location"].values[0]
    destination = locations_df[locations_df["plus_code"] == destination]["location"].values[0]
    travel_time, travel_distance = get_traffic_report(origin, destination)

    new_row = {
        "date": date_now,
        "time": time_now,
        "route_code": route["route_code"],
        "duration": travel_time,
        "distance": travel_distance,
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

driver.quit()

From Kudlu Gate Metro Station, Hosur Road to Biocon Campus, Hosur Road
https://www.google.com/maps/dir/?api=1&origin=Kudlu%20Gate%20Metro%20Station%2C%20Hosur%20Road&destination=Biocon%20Campus%2C%20Hosur%20Road&travelmode=driving
From The Watering Hole, Rajarajeshwari Nagar to Sir Puttanna Chetty Town Hall, Bangalore
https://www.google.com/maps/dir/?api=1&origin=The%20Watering%20Hole%2C%20Rajarajeshwari%20Nagar&destination=Sir%20Puttanna%20Chetty%20Town%20Hall%2C%20Bangalore&travelmode=driving
From Karmelaram Railway Station, Chikkabellandur to St. Francis College, Koramangala
https://www.google.com/maps/dir/?api=1&origin=Karmelaram%20Railway%20Station%2C%20Chikkabellandur&destination=St.%20Francis%20College%2C%20Koramangala&travelmode=driving
From The Rameshwaram Cafe @ Brookfield to Gawky Goose, Wind Tunnel Rd
https://www.google.com/maps/dir/?api=1&origin=The%20Rameshwaram%20Cafe%20%40%20Brookfield&destination=Gawky%20Goose%2C%20Wind%20Tunnel%20Rd&travelmode=driving
From Jaya Prakas

In [6]:
display(df)

,date,time,route_code,duration,distance
0,2025-11-30,01:43,VJRQ+2M|RMJJ+F4,None,None
1,2025-11-30,01:43,WGG8+G5|XH7P+G6,None,None
2,2025-11-30,01:43,WP44+W8|WJFH+XQ,None,None
3,2025-11-30,01:43,XPC7+72|XM33+J3,None,None
4,2025-11-30,01:43,2HM2+P8|XJV5+RG,None,None
5,2025-11-30,01:43,2HVW+G8|XJXR+WG,17 min,9.7 km
6,2025-11-30,01:43,XJPW+92|WJP4+FF,20 min,10.4 km
7,2025-11-30,01:43,XMW9+G8|WMJR+V4,17 min,10.0 km
8,2025-11-30,01:43,WHCJ+26|XGCP+FV,21 min,10.6 km
9,2025-11-30,01:43,WH5F+26|WJ8X+F5W,18 min,9.9 km


In [7]:
def get_duration(s):
    # Handles: "25 min", "1 hr 5 min", "2 hr", "7 min"
    if not isinstance(s, str) or not s.strip():
        return np.nan
    parts = s.split()
    mins = 0
    try:
        if "hr" in parts:
            h_idx = parts.index("hr")
            mins += int(parts[h_idx - 1]) * 60
        if "min" in parts:
            m_idx = parts.index("min")
            mins += int(parts[m_idx - 1])
        # Fallback: if neither token present but a bare integer exists (rare)
        if "hr" not in parts and "min" not in parts:
            mins = float(parts[0])
    except Exception:
        return np.nan
    return mins

# 1) Distance: strip " km", coerce to numeric (invalid -> NaN)
df["distance"] = pd.to_numeric(
    df["distance"].str.replace(" km", "", regex=False), errors="coerce")

# 2) Duration: parse to minutes (invalid -> NaN)
df["duration"] = df["duration"].apply(get_duration)

# 3) Drop rows where either is missing
df = df.dropna(subset=["distance", "duration"]).copy()

# 4) Ensure dtypes
df["distance"] = df["distance"].astype(float)
df["duration"] = df["duration"].astype(int)

In [8]:
df_traffic = transformed_data(df)
display(df_traffic)

,year,month,date,hour,day_of_week,origin,destination,duration,distance,avg_speed
0,2025,11,30,1,Thursday,"Big Bull Temple, Basavanagudi","Shri Someshwara Swamy Temple, Halasuru",23,10.0,26.09
1,2025,11,30,1,Thursday,"RV Road Metro Station, Jayanagar 5th Block","Vijayanagar Metro Station, Chord Road",21,10.6,30.29
2,2025,11,30,1,Thursday,Swami Vivekananda Road Metro Station,"Christ University, Hosur Main Road",20,10.4,31.20
3,2025,11,30,1,Thursday,Jaya Prakash Nagar Metro Station,"Hemavathi Park, HSR Layout",18,9.9,33.00
4,2025,11,30,1,Thursday,"Bethel AG Church, Hebbal",SMVT Railway Station,17,9.7,34.24
5,2025,11,30,1,Thursday,Lulu Mall Bengaluru,Nexus Mall Koramangala,18,10.5,35.00
6,2025,11,30,1,Thursday,Benniganahalli Metro Station,"Embassy TechVillage, Devarabisanahalli",17,10.0,35.29
7,2025,11,30,1,Thursday,"MG Road Metro Station, Bengaluru","Kempegowda International Airport, Bengaluru",39,34.4,52.92


In [9]:
logs = df_traffic[df_traffic['duration'] == df_traffic['duration'].max()]
print(f"{logs['hour'].iloc[0]}hrs [traffic_snapshot] {logs['duration'].iloc[0]} mins @ {logs['avg_speed'].iloc[0]} Km/hr (\u2192 {logs['destination'].iloc[0]})")

1hrs [traffic_snapshot] 39 mins @ 52.92 Km/hr (→ Kempegowda International Airport, Bengaluru)
